In [63]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Embedding, Flatten

from tensorflow.keras.preprocessing.text import Tokenizer

from tensorflow.keras.preprocessing.sequence import pad_sequences

In [64]:
df=pd.read_csv('IMDBDataset.csv')
print(df.info())
df.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   review     50000 non-null  object
 1   sentiment  50000 non-null  object
dtypes: object(2)
memory usage: 781.4+ KB
None


,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [65]:
df.columns

Index(['review', 'sentiment'], dtype='object')

In [66]:
df['sentiment']=df['sentiment'].map({
    'positive':1,
    'negative':0
})

In [67]:
x=df['review']
y=df['sentiment']

In [68]:
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42)

In [69]:
vocab_size = 10000
max_length = 200
tokenizer=Tokenizer(num_words=10000)
tokenizer.fit_on_texts(x_train)

x_train_seq=tokenizer.texts_to_sequences(x_train)
x_test_seq=tokenizer.texts_to_sequences(x_test)

In [70]:
x_train_pad=pad_sequences(x_train_seq,maxlen=200)
x_test_pad=pad_sequences(x_test_seq,maxlen=200)

In [71]:
model=Sequential()

# Add Embedding layer
# input_dim: Vocabulary size (tokenizer.num_words)
# output_dim: Dimension of the dense embedding (e.g., 128)
model.add(Embedding(input_dim=tokenizer.num_words, output_dim=128))

# Flatten the output of the Embedding layer to feed into Dense layers
model.add(Flatten())

model.add(Dense(128,activation='relu'))
model.add(Dense(64,activation='relu'))
model.add(Dense(32,activation='relu'))

model.add(Dense(1,activation='sigmoid'))

In [72]:
model.compile(optimizer='adam',loss='binary_crossentropy',metrics=['accuracy'])
model.summary()

Model: "sequential_7"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_4 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_4 (Flatten)             │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_28 (Dense)                │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_29 (Dense)                │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_30 (Dense)                │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_31 (Dense)                │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [73]:
model.fit(x_train_pad,y_train,epochs=5,batch_size=128,validation_split=0.2)

Epoch 1/5
250/250 ━━━━━━━━━━━━━━━━━━━━ 36s 138ms/step - accuracy: 0.7940 - loss: 0.4136 - val_accuracy: 0.8661 - val_loss: 0.3153
Epoch 2/5
250/250 ━━━━━━━━━━━━━━━━━━━━ 34s 136ms/step - accuracy: 0.9671 - loss: 0.0949 - val_accuracy: 0.8651 - val_loss: 0.3709
Epoch 3/5
250/250 ━━━━━━━━━━━━━━━━━━━━ 35s 142ms/step - accuracy: 0.9960 - loss: 0.0126 - val_accuracy: 0.8551 - val_loss: 0.6803
Epoch 4/5
250/250 ━━━━━━━━━━━━━━━━━━━━ 34s 138ms/step - accuracy: 0.9957 - loss: 0.0121 - val_accuracy: 0.8549 - val_loss: 0.7188
Epoch 5/5
250/250 ━━━━━━━━━━━━━━━━━━━━ 42s 142ms/step - accuracy: 0.9953 - loss: 0.0133 - val_accuracy: 0.8560 - val_loss: 0.6774


In [74]:
loss,accuracy=model.evaluate(x_test_pad,y_test)
print("Test Accuracy: ",accuracy)

313/313 ━━━━━━━━━━━━━━━━━━━━ 3s 9ms/step - accuracy: 0.8556 - loss: 0.6592
Test Accuracy:  0.8555999994277954


In [77]:
review=['This movie was fabulous']
review_seq=tokenizer.texts_to_sequences(review)
review_pad=pad_sequences(review_seq,maxlen=200)
prediction=model.predict(review_pad)
if(prediction>0.5):
    print("Positive Review")
else:
    print("Negative Review")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step
Positive Review
